# Deep Learning Image Classification

A clean portfolio version of the original notebook for a binary image classification task.

The project compares:
- A custom CNN baseline
- MobileNetV2
- VGG16
- ResNet50

The transfer-learning models use ImageNet pretrained weights and are implemented with TensorFlow/Keras.

> Dataset files are not included in this repository. Update `DATA_DIR` before running the notebook.


## 1. Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import warnings
warnings.filterwarnings("ignore")


## 2. Configuration

In [ ]:
# Update this path after downloading/preparing the dataset.
DATA_DIR = Path("./data")

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "valid"
TEST_DIR = DATA_DIR / "test"

MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 2
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


## 3. Dataset indexing

Expected folder structure:

```text
data/
├── train/
│   ├── Class0/
│   └── Class1/
├── valid/
│   ├── Class0/
│   └── Class1/
└── test/
    ├── Class0/
    └── Class1/
```


In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def create_dataframe_from_dir(directory):
    directory = Path(directory)

    if not directory.exists():
        raise FileNotFoundError(f"Dataset directory not found: {directory}")

    filepaths = []
    labels = []

    for class_dir in sorted(directory.iterdir()):
        if not class_dir.is_dir():
            continue

        for file_path in sorted(class_dir.iterdir()):
            if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS:
                filepaths.append(str(file_path))
                labels.append(class_dir.name)

    if not filepaths:
        raise ValueError(f"No supported image files found in: {directory}")

    return pd.DataFrame({
        "filepath": filepaths,
        "label": labels,
    })


df_train = create_dataframe_from_dir(TRAIN_DIR)
df_val = create_dataframe_from_dir(VAL_DIR)
df_test = create_dataframe_from_dir(TEST_DIR)


## 4. Image preprocessing and data generators

In [ ]:
# The original notebook used rescaling only.
train_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
eval_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_gen = train_datagen.flow_from_dataframe(
    dataframe=df_train,
    x_col="filepath",
    y_col="label",
    target_size=IMAGE_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

valid_gen = eval_datagen.flow_from_dataframe(
    dataframe=df_val,
    x_col="filepath",
    y_col="label",
    target_size=IMAGE_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_gen = eval_datagen.flow_from_dataframe(
    dataframe=df_test,
    x_col="filepath",
    y_col="label",
    target_size=IMAGE_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
)


## 5. Custom CNN baseline

In [ ]:
def build_custom_cnn():
    model = Sequential([
        Conv2D(
            8,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
            input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
        ),
        Conv2D(
            16,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
        ),
        MaxPooling2D((2, 2)),
        Conv2D(
            32,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
        ),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.2),
        Dense(NUM_CLASSES, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


custom_cnn = build_custom_cnn()


## 6. Transfer learning models

### 6.1 MobileNetV2

In [ ]:
def build_mobilenet_v2():
    base_model = tf.keras.applications.MobileNetV2(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
        pooling="avg",
    )

    # Preserve the fine-tuning strategy from the original notebook.
    for layer in base_model.layers[:140]:
        layer.trainable = False

    for layer in base_model.layers[140:]:
        layer.trainable = True

    model = Sequential([
        base_model,
        Dropout(0.2),
        Dense(NUM_CLASSES, activation="softmax"),
    ])

    model.compile(
        optimizer=Adamax(learning_rate=1e-6),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


mobilenet_v2 = build_mobilenet_v2()


### 6.2 VGG16

In [ ]:
def build_vgg16():
    base_model = keras.applications.VGG16(
        include_top=False,
        weights="imagenet",
        pooling="avg",
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    )

    for layer in base_model.layers:
        layer.trainable = False

    # Fine-tune the last three layers, following the original notebook.
    for layer in base_model.layers[-3:]:
        layer.trainable = True

    model = Sequential([
        base_model,
        Dropout(0.3),
        Dense(NUM_CLASSES, activation="softmax"),
    ])

    model.compile(
        optimizer=Adamax(learning_rate=1e-6),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


vgg16 = build_vgg16()


### 6.3 ResNet50

In [ ]:
def build_resnet50():
    base_model = keras.applications.ResNet50(
        include_top=False,
        weights="imagenet",
        pooling="max",
        input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    )

    model = Sequential([
        base_model,
        Dense(NUM_CLASSES, activation="softmax"),
    ])

    model.compile(
        optimizer=Adamax(learning_rate=1e-6),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )

    return model


resnet50 = build_resnet50()


## 7. Training

In [ ]:
def train_model(
    model,
    model_name,
    epochs=30,
    use_early_stopping=True,
):
    callbacks = []

    if use_early_stopping:
        callbacks.append(
            EarlyStopping(
                monitor="val_loss",
                patience=3,
                restore_best_weights=True,
            )
        )

    history = model.fit(
        train_gen,
        validation_data=valid_gen,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )

    model.save(MODEL_DIR / f"{model_name}.keras")

    return history


In [ ]:
# Uncomment the models you want to train.

# history_custom = train_model(
#     custom_cnn,
#     "custom_cnn",
#     epochs=10,
#     use_early_stopping=False,
# )

# history_mobilenet = train_model(
#     mobilenet_v2,
#     "mobilenet_v2",
#     epochs=100,
# )

# history_vgg16 = train_model(
#     vgg16,
#     "vgg16",
#     epochs=100,
# )

# history_resnet50 = train_model(
#     resnet50,
#     "resnet50",
#     epochs=100,
# )


## 8. Model evaluation

This helper keeps the evaluation code compact and returns metrics without generating charts.


In [ ]:
def evaluate_model(model, generator):
    generator.reset()

    probabilities = model.predict(generator, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    true_labels = generator.classes

    return {
        "accuracy": accuracy_score(true_labels, predictions),
        "precision_macro": precision_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            true_labels,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }


# Example:
# vgg16_metrics = evaluate_model(vgg16, test_gen)


## 9. Single-image inference

In [ ]:
def predict_image(model, image_path, class_indices):
    image = Image.open(image_path).convert("RGB")
    image = image.resize(IMAGE_SIZE)

    array = np.asarray(image, dtype=np.float32) / 255.0
    array = np.expand_dims(array, axis=0)

    probabilities = model.predict(array, verbose=0)[0]

    index_to_class = {
        index: class_name
        for class_name, index in class_indices.items()
    }

    predicted_index = int(np.argmax(probabilities))

    return {
        "predicted_class": index_to_class[predicted_index],
        "confidence": float(probabilities[predicted_index]),
        "probabilities": {
            index_to_class[i]: float(probabilities[i])
            for i in range(len(probabilities))
        },
    }


# Example:
# result = predict_image(
#     vgg16,
#     "./data/test/Class0/example.jpg",
#     test_gen.class_indices,
# )


## 10. Notes for GitHub

- Keep the dataset outside Git tracking.
- Add `data/` and `models/` to `.gitignore`.
- The original notebook contained training outputs, charts, repeated evaluation cells and Google Drive paths; these have been removed from this portfolio version.
- Replace `Class0` and `Class1` with the real class names in your README if the dataset has meaningful labels.
